In [14]:
import tensorflow as tf
from tensorflow.keras import layers 
from tensorflow.keras.preprocessing.text import Tokenizer 
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [15]:
texts = [
    # Positive
    "This phone is excellent",
    "Amazing battery life",
    "Very fast performance",
    "Camera quality is great",
    "I love this laptop",
    "Absolutely outstanding product",
    "Best purchase I have ever made",
    "Incredible sound quality",
    "This device exceeded my expectations",
    "Superb build quality and design",
    # Negative
    "Terrible customer service",
    "Battery drains quickly",
    "Very slow device",
    "Worst phone ever",
    "I hate this product",
    "Completely broken after one week",
    "Very disappointed with this purchase",
    "Awful screen and poor display",
    "This product stopped working immediately",
    "Terrible value for money",
    # Neutral
    "I like this product but it's not the best",
    "It works fine nothing special",
    "Average performance for the price",
    "Decent product but could be improved",
    "Not bad but not great either",
    "It does the job nothing more",
    "Some features are good some are bad",
    "Okay product with mixed results",
    "Neither impressed nor disappointed",
    "Pretty standard device overall",
]

In [16]:
# 1 = Positive
# 0 = Negative
# 2 = Neutral

labels = [
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  # Positive
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  # Negative
    2, 2, 2, 2, 2, 2, 2, 2, 2, 2   # Neutral
]

In [17]:
vocab_size = 1000
max_length = 8

In [18]:
tokenizer = Tokenizer()
tokenizer = Tokenizer(num_words = vocab_size, oov_token = "<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen = max_length, padding = "post")

print("Word Index:")
print(tokenizer.word_index)

Word Index:
{'<OOV>': 1, 'this': 2, 'product': 3, 'i': 4, 'very': 5, 'quality': 6, 'device': 7, 'but': 8, 'not': 9, 'the': 10, 'phone': 11, 'is': 12, 'battery': 13, 'performance': 14, 'great': 15, 'best': 16, 'purchase': 17, 'ever': 18, 'and': 19, 'terrible': 20, 'disappointed': 21, 'with': 22, 'for': 23, 'it': 24, 'nothing': 25, 'bad': 26, 'some': 27, 'are': 28, 'excellent': 29, 'amazing': 30, 'life': 31, 'fast': 32, 'camera': 33, 'love': 34, 'laptop': 35, 'absolutely': 36, 'outstanding': 37, 'have': 38, 'made': 39, 'incredible': 40, 'sound': 41, 'exceeded': 42, 'my': 43, 'expectations': 44, 'superb': 45, 'build': 46, 'design': 47, 'customer': 48, 'service': 49, 'drains': 50, 'quickly': 51, 'slow': 52, 'worst': 53, 'hate': 54, 'completely': 55, 'broken': 56, 'after': 57, 'one': 58, 'week': 59, 'awful': 60, 'screen': 61, 'poor': 62, 'display': 63, 'stopped': 64, 'working': 65, 'immediately': 66, 'value': 67, 'money': 68, 'like': 69, "it's": 70, 'works': 71, 'fine': 72, 'special': 73, '

In [19]:
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()

        
        """ 1. Token Embedding
            Converts each word's integer ID into a 16-dimensional vector.
            The vectors are learned during training.

        """
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
        

        """ 2. Positional Embedding
            Gives the model a sense of word order.
            Transformers process all words at once, not sequentially like RNNs.
            Generates position indices, then adds position vectors on top of token vectors.
        """
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [20]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()


        """ 3. Multi-Head Attention
                Lets every word look at every other word in the sequence and decide how much to "pay attention" to each one.
                num_heads=2 means it does this from 2 different perspectives simultaneously, then combines the results.
        """
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )


        """ 4. Feed Forward Network
                Processes each token's representation independently through 2 Dense layers.
                First layer expands from 16 - 32 dims.
                Second layer compresses back from 32 - 16 dims.
                Attention finds relationships between words,

        """
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence
        attention_output = self.attention(inputs, inputs)


        """ 5. Add & Normalize
                First Add & Norm — after attention: + adds the original input back to the attention output. This prevents vanishing gradients and helps the model
                retain information.
                Second Add & Norm — after FFN: preserves what was learned before FFN, then normalize again to keep values going to the next layer.
        """
        out1 = self.layernorm1(inputs + attention_output)
        # Feed-forward network
        ffn_output = self.ffn(out1)
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

In [21]:
# build the model
embed_dim = 16
num_heads = 2
ff_dim = 32
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 

""" 6. Output Layer
        GlobalAveragePooling1D collapses the sequence dimension, the entire sentence into one fixed-size representation.
        The model needs a single scalar prediction per sentence, not 20 separate vectors. Pooling summarises the whole sequence; 
        sigmoid converts the score to a probability.

"""
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(3, activation="softmax")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [22]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]

)

In [23]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_1  │ (None, 8, 16)          │        16,128 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 8, 16)          │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,475 (76.07 KB)

 Trainable params: 19,475 (76.07 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
import numpy as np

X = pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=max_length, padding="post")
X = np.array(X)        # ← add this line

labels = np.array(labels)   # ← also convert labels

model.fit(
    X,
    labels,
    epochs=30, 
    batch_size= 2,
    verbose=1
)

Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2667 - loss: 1.2227
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5333 - loss: 1.0139
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7000 - loss: 0.8398
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8333 - loss: 0.6150 
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9333 - loss: 0.4888 
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.3325 
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.2620 
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.1795 
Epoch 9/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.1392 
Epoch 10/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.1079 
Epoch 11/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0768 
Epoch 12/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0

In [25]:
loss, accuracy = model.evaluate(X, labels)

print("Accuracy:", accuracy)  

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 1.0000 - loss: 0.0043
Accuracy: 1.0


In [26]:
test_sentences = ["I love the film", "This movie was awful", "It was okay nothing special"]
X_test = pad_sequences(tokenizer.texts_to_sequences(test_sentences),
                        maxlen=max_length, padding="post")

predictions = model.predict(X_test)
classes = ["Negative", "Positive", "Neutral"]

for sentence, pred in zip(test_sentences, predictions):
    idx = np.argmax(pred)
    confidence = pred[idx] * 100
    print(f"{sentence}")
    print(f"  → {classes[idx]} ({confidence:.1f}% confidence)")
    print(f"     [Neg: {pred[0]:.2f}  Pos: {pred[1]:.2f}  Neu: {pred[2]:.2f}]\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
I love the film
  → Positive (69.7% confidence)
     [Neg: 0.03  Pos: 0.70  Neu: 0.27]

This movie was awful
  → Negative (99.4% confidence)
     [Neg: 0.99  Pos: 0.00  Neu: 0.00]

It was okay nothing special
  → Neutral (99.3% confidence)
     [Neg: 0.00  Pos: 0.00  Neu: 0.99]

